In [ ]:
!pip install langchain_community
!pip install requests==2.32.4 --force-reinstall# pypdfloader

In [ ]:
!pip install langchain_experimental

In [ ]:
!pip install pypdf
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader('/content/llm.pdf')
pages=loader.load()
# print(pages[0].page_content)

print(len(pages))
print(pages[1].metadata)
key_metadata=['description-abstract','producer','eventtype','created','creator','date','creationdate','firstpage','lastpage','subject','language','moddate','description','editors']


for doc in pages:
    for key in key_metadata:
        doc.metadata.pop(key,None)

In [ ]:
!pip install langchain_mistralai
from langchain_experimental.text_splitter import SemanticChunker
from langchain_mistralai import MistralAIEmbeddings
MISTRAL_API_KEY="CrZMBjlCHzQ2eWVY69UYcqJ24HBHkGmS"
embeddings = MistralAIEmbeddings(
    model="mistral-embed",
    api_key=MISTRAL_API_KEY
)

splitterS = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=80
)

semantic_chunks = splitterS.split_documents(pages)

In [ ]:
!pip install sentence_transformers

In [ ]:
from sentence_transformers import CrossEncoder

reranker1= CrossEncoder("BAAI/bge-reranker-base")


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

In [ ]:
!pip install redis
!pip install redisvl
!pip install langchain-redis

In [ ]:
# from google.colab import userdata
# REDIS_PASSWORD="hAV5SAKpBN9pVwoXoqyZQSDZLWtEAaVr"
# REDIS_HOST = "redis-16443.c264.ap-south-1-1.ec2.cloud.redislabs.com"
# REDIS_PORT = 16443
# REDIS_PASSWORD = userdata.get('REDIS_PASSWORD')  # set this in Colab's 🔑 Secrets panel
REDIS_HOST = "0.tcp.in.ngrok.io"
REDIS_PORT = 20629
import redis, time

r = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    # password=password",
    decode_responses=True
)

t0 = time.perf_counter()
print(r.ping())
t1 = time.perf_counter()
print(f"Round trip: {(t1-t0)*1000:.1f} ms")



In [ ]:
from langchain_redis import RedisVectorStore
redis_url = "redis://0.tcp.in.ngrok.io:20629"
vectorstore = RedisVectorStore(
    embeddings=embeddings,
    redis_url=redis_url,
    index_name="llm_paper"
)

from langchain_redis import RedisVectorStore
redis_url = "redis://0.tcp.in.ngrok.io:20629"
semantic_cach = RedisVectorStore(
    embeddings=embeddings,
    redis_url=redis_url,
    index_name="rag_cach"
)

In [ ]:
MISTRAL_API_KEY="CrZMBjlCHzQ2eWVY69UYcqJ24HBHkGmS"
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model="mistral-small-latest",
    max_tokens=100,
    temperature=0.4,
    api_key=MISTRAL_API_KEY
)

In [ ]:
MISTRAL_API_KEY="CrZMBjlCHzQ2eWVY69UYcqJ24HBHkGmS"
from langchain_mistralai import ChatMistralAI

llm1= ChatMistralAI(
    model="mistral-small-latest",
    max_tokens=250,
    temperature=0.4,
    api_key=MISTRAL_API_KEY
)

In [ ]:
from langchain_core.documents import Document
def store_answer_in_cache(question: str, answer: str):
    semantic_cach.add_documents([
        Document(page_content=question, metadata={"answer": answer})
    ])

In [ ]:
import time

def rag_pipeline(question):
    # ------------------ Total Pipeline ------------------
    pipeline_start = time.perf_counter()

    # ------------------ Query Transformation ------------------
    qt_start = time.perf_counter()

    rewrite_prompt = f"""
Rewrite the following question so that it is better suited for semantic search.

Question:
{question}

Rewritten Query:
"""

    def needs_rewrite(q: str) -> bool:
        return len(q.split()) > 12

    if needs_rewrite(question):
        rewritten_query = llm.invoke(rewrite_prompt).content.strip()
    else:
        rewritten_query = question

    qt_end = time.perf_counter()


    # ------------------ Redis Retrieval ------------------
    retrieval_start = time.perf_counter()

    results3 = vectorstore.similarity_search(
        query=rewritten_query,
        k=5
    )

    retrieval_end = time.perf_counter()

    # ------------------ Prepare Pairs ------------------
    pair_start = time.perf_counter()

    pairs = [
        [rewritten_query, doc.page_content]
        for doc in results3
    ]

    pair_end = time.perf_counter()

    # ------------------ Cross Encoder ------------------
    rerank_start = time.perf_counter()

    scores = reranker1.predict(pairs)

    ranked = sorted(
        zip(results3, scores),
        key=lambda x: x[1],
        reverse=True
    )

    results = [doc for doc, score in ranked[:5]]

    rerank_end = time.perf_counter()

    # ------------------ Context Creation ------------------
    context_start = time.perf_counter()

    context = "\n\n".join(
        f"""
Source: {doc.metadata.get('source')}
Page: {doc.metadata.get('page')}

{doc.page_content}
"""
        for doc in results
    )

    prompt = f"""
Answer only from the context below with proper resources in brifly and with proper formating.
If the answer is not present, say "I don't have the answer."

Context:
{context}

Question:
{question}
"""

    context_end = time.perf_counter()

    # ------------------ LLM ------------------
    llm_start = time.perf_counter()

    answer = llm.invoke(prompt)

    llm_end = time.perf_counter()

    # ------------------ Total ------------------
    pipeline_end = time.perf_counter()

    store_answer_in_cache(question, answer.content)

    print("=" * 50)
    print(f"Query Transformation : {qt_end - qt_start:.3f} sec")
    print(f"Redis Retrieval      : {retrieval_end - retrieval_start:.3f} sec")
    print(f"Pair Creation        : {pair_end - pair_start:.3f} sec")
    print(f"Cross Encoder        : {rerank_end - rerank_start:.3f} sec")
    print(f"Context Creation     : {context_end - context_start:.3f} sec")
    print(f"LLM Generation       : {llm_end - llm_start:.3f} sec")
    print("-" * 50)
    print(f"Total Pipeline       : {pipeline_end - pipeline_start:.3f} sec")
    print("=" * 50)

    print("\nAnswer:\n")
    print(answer.content)

    return answer.content



def ask(question):
    results = semantic_cach.similarity_search_with_score(query=question, k=1)

    if results:
        doc, score = results[0]
        print(f"Cache Score: {score}")

        if score < 0.08:
            output = doc.metadata["answer"]
            # prompt = f"answer it {output}"
            # return llm1.invoke(prompt).content
            return output

    # Cache miss or no results
    return rag_pipeline(question)




In [ ]:
answer = ask("explain query,key,value")
print(answer)